# Standard molecular docking workflow | AutoDock Vina 1.2.3
A reusable, step-by-step Google Colab workflow based on the taurine–aging study. Run cells in order. Upload one protein and one ligand from your computer or Google Drive. The docking search conditions are fixed to the manuscript; the receptor, ligand and binding site are configurable. For exact reproduction use the original prepared PDBQT files, original analysis receptor PDB and original grid/CPU settings. Newly prepared structures may yield different results. This notebook is a reusable extension, not proof that all newly implemented preparation and analysis steps were used in the original study.

## 1. Install and verify AutoDock Vina 1.2.3
The official Linux x86-64 executable is version-checked. Other packages are recorded at runtime; their historical versions have not been confirmed. Open Babel is optional and is needed for the original STAT5B preparation route.

In [ ]:
#@title Install dependencies
import sys, subprocess, shutil, importlib.metadata as im, urllib.request, platform
from pathlib import Path
import json, hashlib, re, math, datetime
subprocess.run([sys.executable,'-m','pip','install','-q','meeko','rdkit','gemmi','py3Dmol'],check=True)
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
import py3Dmol
from IPython.display import display
if platform.machine() != 'x86_64': raise RuntimeError('Use an x86-64 Linux Colab runtime.')
VINA = Path('/content/vina_1.2.3')
URL = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.3/vina_1.2.3_linux_x86_64'
if not VINA.exists(): urllib.request.urlretrieve(URL,VINA)
VINA.chmod(0o755)
version = subprocess.check_output([str(VINA),'--version'],text=True,stderr=subprocess.STDOUT).strip()
if not re.search(r'\b1\.2\.3\b',version): raise RuntimeError(version)
VERSIONS = {p:im.version(p) for p in ['meeko','rdkit','gemmi','py3Dmol','numpy','pandas']}
VERSIONS.update(vina=version,vina_sha256=hashlib.sha256(VINA.read_bytes()).hexdigest())
print(VERSIONS)
for exe in ['mk_prepare_receptor.py','mk_prepare_ligand.py']:
    if not shutil.which(exe): raise RuntimeError(f'Missing {exe}')

## 2. Select input source and create a run folder
For Google Drive, enter full paths in the next two cells, such as `/content/drive/MyDrive/Docking/protein.pdb`. Outputs are written to a unique run folder; optional Drive saving is at the end.

In [ ]:
#@title Set input source
INPUT_SOURCE = "Upload from computer" #@param ["Upload from computer", "Google Drive"]
PROJECT_NAME = "molecular_docking" #@param {type:"string"}
from google.colab import files, drive
if INPUT_SOURCE == 'Google Drive': drive.mount('/content/drive')
name = re.sub(r'[^a-zA-Z0-9_-]+', '_', PROJECT_NAME).strip('_') or 'docking'
RUN = Path('/content/docking_runs') / (name + '_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
for sub in ['inputs','prepared','results','logs']: (RUN/sub).mkdir(parents=True, exist_ok=True)
def get_input(label, suffixes, drive_path=''):
    if INPUT_SOURCE == 'Upload from computer':
        print(f'Select one {label} file: {suffixes}')
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError('Select exactly one file.')
        filename, data=next(iter(uploaded.items())); source_name=Path(filename).name
    else:
        p=Path(drive_path)
        if not p.is_file(): raise FileNotFoundError(drive_path)
        source_name=p.name; data=p.read_bytes()
    if Path(source_name).suffix.lower() not in suffixes: raise ValueError(f'Expected {suffixes}, got {source_name}')
    out=RUN/'inputs'/(label+'_'+source_name); out.write_bytes(data)
    return out

def command(args, log):
    proc=subprocess.run([str(x) for x in args], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log.write_text(proc.stdout)
    print(proc.stdout)
    if proc.returncode: raise RuntimeError(f'Preparation failed; see {log.name}')
print(RUN)

## 3. Upload receptor
For manuscript reproduction, supply the original prepared rigid receptor PDBQT and optionally the original PDB for interaction analysis. For a new receptor, supply a reviewed PDB with correct protonation, hydrogens, missing atoms and any necessary cofactors. Chain A is the study default; choose the appropriate chain for other proteins.

In [ ]:
#@title Protein file
PROTEIN_PATH = "" #@param {type:"string"}
protein = get_input('protein', {'.pdb','.pdbqt'}, PROTEIN_PATH)
print(protein.name)

## 4. Upload ligand
Use the intended stereochemistry and protonation/charge state. SDF must contain one valid 3D molecule. For taurine, verify the exact chemical state used in the study; no automatic pH conversion is performed.

In [ ]:
#@title Ligand file
LIGAND_PATH = "" #@param {type:"string"}
ligand = get_input('ligand', {'.sdf','.pdbqt'}, LIGAND_PATH)
print(ligand.name)

## 5. Prepare receptor and ligand
The PDB route selects the requested chain and removes only explicitly listed hetero-residue names. Inspect the result and resolve alternate locations, missing atoms, protonation and cofactors before preparation. Original PDBQT files bypass preparation. Ligands must have a valid 3D structure and the intended protonation state. For taurine select the zwitterionic verification option. Meeko is the default; the Open Babel route mirrors the STAT5B ligand preparation commands. Preparation of a new receptor with Meeko is a reusable alternative, not a verified reconstruction of the historical preparation procedure.

In [ ]:
#@title Prepare receptor
CHAIN = "A" #@param {type:"string"}
REMOVE_HETERO_RESNAMES = "HOH,WAT,SOL" #@param {type:"string"}
RECEPTOR_PDB_REVIEWED = False #@param {type:"boolean"}
rec=RUN/'prepared'/'receptor.pdbqt'
rec.unlink(missing_ok=True)
analysis_pdb=None
if protein.suffix.lower()=='.pdbqt':
    shutil.copy2(protein,rec)
else:
    if not RECEPTOR_PDB_REVIEWED: raise ValueError('Review receptor chemistry and enter the non-essential hetero-residue names to remove; then confirm.')
    removed={s.strip().upper() for s in REMOVE_HETERO_RESNAMES.split(',') if s.strip()}
    raw=protein.read_text().splitlines()
    if sum(l.startswith('MODEL') for l in raw)>1: raise ValueError('Supply a single structural model.')
    kept=[l for l in raw if l.startswith(('ATOM  ','HETATM')) and (not CHAIN or l[21:22]==CHAIN) and not (l.startswith('HETATM') and l[17:20].strip().upper() in removed)]
    if not any(l.startswith('ATOM  ') for l in kept): raise ValueError('No protein atoms for requested chain.')
    if any(l[16:17] not in (' ','') for l in kept): raise ValueError('Resolve alternate atom locations before running.')
    analysis_pdb=RUN/'prepared'/'reviewed_receptor.pdb'
    analysis_pdb.write_text('\n'.join(kept)+'\nEND\n')
    command(['mk_prepare_receptor.py','--read_pdb',analysis_pdb,'-o',RUN/'prepared'/'receptor','-p'],RUN/'logs'/'receptor_preparation.log')
if not rec.is_file() or not rec.stat().st_size: raise RuntimeError('No receptor PDBQT produced.')
print(rec)

In [ ]:
#@title Prepare ligand
LIGAND_TOOL = "Meeko" #@param ["Meeko","Open Babel"]
VERIFY_TAURINE = True #@param {type:"boolean"}
lig=RUN/'prepared'/'ligand.pdbqt'
lig.unlink(missing_ok=True)
if ligand.suffix.lower()=='.pdbqt':
    shutil.copy2(ligand,lig)
    print('Prepared PDBQT supplied: chemical state must be verified from original ligand structure.')
else:
    mols=list(Chem.SDMolSupplier(str(ligand),removeHs=False))
    if len(mols)!=1 or mols[0] is None: raise ValueError('Supply one valid SDF molecule.')
    mol=mols[0]
    if mol.GetNumConformers()!=1 or not mol.GetConformer().Is3D(): raise ValueError('Supply a verified 3D SDF.')
    if VERIFY_TAURINE:
        expected=Chem.MolToSmiles(Chem.MolFromSmiles('[NH3+]CCS(=O)(=O)[O-]'))
        if Chem.MolToSmiles(Chem.RemoveHs(mol))!=expected or rdMolDescriptors.CalcMolFormula(mol)!='C2H7NO3S': raise ValueError('Expected zwitterionic taurine; disable verification for another ligand.')
    print('Formula:',rdMolDescriptors.CalcMolFormula(mol),'formal charge:',Chem.GetFormalCharge(mol),'SMILES:',Chem.MolToSmiles(mol))
    if LIGAND_TOOL=='Meeko':
        mol=Chem.AddHs(mol,addCoords=True)
        sdf=RUN/'prepared'/'ligand_with_hydrogens.sdf'
        with Chem.SDWriter(str(sdf)) as w: w.write(mol)
        command(['mk_prepare_ligand.py','-i',sdf,'-o',lig],RUN/'logs'/'ligand_preparation.log')
    else:
        if not shutil.which('obabel'):
            subprocess.run(['apt-get','update','-qq'],check=True)
            subprocess.run(['apt-get','install','-y','openbabel'],check=True)
        VERSIONS['openbabel']=subprocess.check_output(['obabel','-V'],text=True).strip()
        sdf=RUN/'prepared'/'ligand_openbabel.sdf'
        command(['obabel',ligand,'-O',sdf,'-h','--gen3d'],RUN/'logs'/'openbabel_3d.log')
        command(['obabel',sdf,'-O',lig,'-xh'],RUN/'logs'/'ligand_preparation.log')
if not lig.is_file() or not lig.stat().st_size: raise RuntimeError('No ligand PDBQT produced.')
print(lig)

## 6. Define the docking box (Å)
Select manual coordinates, reference ligand (in the same coordinate frame), or specified pocket residues. Presets are only for the original study structures and conformations, not arbitrary PDBs. The bounding-box padding for new inputs is configurable and is not a historical study parameter. Inspect the box before docking.

In [ ]:
#@title Docking box options
BOX_MODE = "Manual coordinates" #@param ["Manual coordinates", "Reference ligand", "Pocket residues", "Paper preset"]
PAPER_TARGET = "MAPK3" #@param ["MAPK3","EGFR","STAT5A","STAT5B"]
POCKET_RESIDUES = "A:51,A:52" #@param {type:"string"}
PRESETS={'MAPK3':([28.616,7.003,18.087],[20,20,22],0),'EGFR':([18.866,35.252,37.650],[22,22,23.2],0),'STAT5A':([22.435,59.146,20.009],[20.332,26.288,25.497],8),'STAT5B':([29.244,8.171,91.108],[22,22,22],0)}
CENTER_X = 0.0 #@param {type:"number"}
CENTER_Y = 0.0 #@param {type:"number"}
CENTER_Z = 0.0 #@param {type:"number"}
SIZE_X = 20.0 #@param {type:"number"}
SIZE_Y = 20.0 #@param {type:"number"}
SIZE_Z = 20.0 #@param {type:"number"}
REFERENCE_PATH = "" #@param {type:"string"}
PADDING = 6.0 #@param {type:"number"}

def pdb_atoms(path):
    atoms=[]
    for line in Path(path).read_text().splitlines():
        if line.startswith(('ATOM  ','HETATM')):
            xyz=[float(line[30:38]),float(line[38:46]),float(line[46:54])]
            atoms.append((line,xyz))
    if not atoms: raise ValueError(f'No atoms found: {path}')
    return atoms

def bounding_box(xyz,padding):
    xyz=np.asarray(xyz,dtype=float)
    if not np.isfinite(padding) or padding<=0: raise ValueError('Padding must be positive.')
    return (xyz.max(0)+xyz.min(0))/2, xyz.max(0)-xyz.min(0)+2*padding

rxyz=np.array([xyz for _,xyz in pdb_atoms(rec)])
reference=None
if BOX_MODE=='Manual coordinates':
    center=np.array([CENTER_X,CENTER_Y,CENTER_Z]); size=np.array([SIZE_X,SIZE_Y,SIZE_Z])
elif BOX_MODE=='Paper preset':
    center,size=map(lambda a: np.array(a,dtype=float),PRESETS[PAPER_TARGET][:2])
elif BOX_MODE=='Pocket residues':
    wanted={x.strip() for x in POCKET_RESIDUES.split(',') if x.strip()}
    matches={f'{line[21:22]}:{line[22:26].strip()}{line[26:27].strip()}' for line,_ in pdb_atoms(rec)}
    if not wanted or not wanted<=matches: raise ValueError(f'Pocket residues not found: {wanted-matches}')
    xyz=[xyz for line,xyz in pdb_atoms(rec) if f'{line[21:22]}:{line[22:26].strip()}{line[26:27].strip()}' in wanted]
    center,size=bounding_box(xyz,PADDING)
else:
    reference=get_input('reference', {'.pdb','.pdbqt','.sdf'},REFERENCE_PATH)
    if reference.suffix.lower()=='.sdf':
        ref_mols=list(Chem.SDMolSupplier(str(reference),removeHs=False))
        if len(ref_mols)!=1 or ref_mols[0] is None or not ref_mols[0].GetNumConformers() or not ref_mols[0].GetConformer().Is3D(): raise ValueError('Reference must be one 3D structure.')
        refxyz=ref_mols[0].GetConformer().GetPositions()
    else: refxyz=np.array([xyz for _,xyz in pdb_atoms(reference)])
    center,size=bounding_box(refxyz,PADDING)
if not np.all(np.isfinite(center)) or not np.all(np.isfinite(size)) or np.any(size<=0): raise ValueError('Invalid box.')
if not np.any(np.all(np.abs(rxyz-center)<=size/2,axis=1)): raise ValueError('Box does not contain any receptor atoms; check coordinate system.')
print('Center (Å):',center,'Size (Å):',size,'Volume (Å³):',np.prod(size))
if np.prod(size)>27000: print('WARNING: large search space. Consider a focused binding site and search convergence.')

def show_structure(receptor, pose=None, box=False):
    view=py3Dmol.view(width=800,height=480)
    view.addModel(Path(receptor).read_text(),'pdbqt' if Path(receptor).suffix.lower()=='.pdbqt' else 'pdb')
    view.setStyle({'model':0},{'cartoon':{'color':'spectrum'},'stick':{'radius':0.12}})
    if pose:
        view.addModel(Path(pose).read_text(),'pdbqt'); view.setStyle({'model':1},{'stick':{'colorscheme':'greenCarbon','radius':0.22}})
    if box:
        view.addBox({'center':dict(zip('xyz',map(float,center))),'dimensions':{'w':float(size[0]),'h':float(size[1]),'d':float(size[2])},'color':'blue','opacity':0.2,'wireframe':True})
    view.zoomTo(); view.show()
show_structure(rec,box=True)

## 7. Confirm fixed manuscript docking settings
The Vina search settings are fixed. CPU is automatically set to the historical value for a study preset (8 for STAT5A, 0 = automatic for the others). Exact output also depends on input preparation, binary and runtime environment.

In [ ]:
#@title Confirm standard docking settings
EXHAUSTIVENESS=32
NUM_POSES=20
ENERGY_RANGE=5.0
SEED=20260906
CPU=PRESETS[PAPER_TARGET][2] if BOX_MODE=='Paper preset' else 0
BOX_CONFIRMED = False #@param {type:"boolean"}
if not BOX_CONFIRMED: raise ValueError('Inspect and confirm the box.')
settings={'scoring':'vina','center_A':center.tolist(),'size_A':size.tolist(),'box_mode':BOX_MODE,'paper_target':PAPER_TARGET if BOX_MODE=='Paper preset' else None,'pocket_residues':POCKET_RESIDUES if BOX_MODE=='Pocket residues' else None,'padding_A':PADDING if BOX_MODE in ('Pocket residues','Reference ligand') else None,'exhaustiveness':EXHAUSTIVENESS,'num_modes':NUM_POSES,'energy_range_kcal_mol':ENERGY_RANGE,'seed':SEED,'cpu':CPU,'versions':VERSIONS,'python':sys.version,'chain':CHAIN if protein.suffix.lower()=='.pdb' else None,'removed_hetero_resnames':REMOVE_HETERO_RESNAMES if protein.suffix.lower()=='.pdb' else None,'ligand_preparation':LIGAND_TOOL if ligand.suffix.lower()=='.sdf' else 'original PDBQT','sha256':{str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [protein,ligand,rec,lig]+([reference] if reference else [])}}
(RUN/'results'/'docking_settings.json').write_text(json.dumps(settings,indent=2))
print(json.dumps(settings,indent=2))

## 8. Run docking
Use the same Vina command-line interface as the original workflow. The software can return fewer than 20 poses. Preserve the full output and log.

In [ ]:
#@title Execute AutoDock Vina 1.2.3
if not BOX_CONFIRMED: raise ValueError('Confirm box.')
if settings['center_A']!=center.tolist() or settings['size_A']!=size.tolist() or any(hashlib.sha256((RUN/p).read_bytes()).hexdigest()!=h for p,h in settings['sha256'].items()): raise ValueError('Inputs or box changed: rerun settings.')
if (EXHAUSTIVENESS,NUM_POSES,ENERGY_RANGE,SEED)!=(32,20,5.0,20260906): raise ValueError('Standard parameters changed.')
docked=RUN/'results'/'docked_poses.pdbqt'
docked.unlink(missing_ok=True)
config=RUN/'results'/'vina_config.txt'
params={'receptor':rec,'ligand':lig,**{f'center_{a}':float(v) for a,v in zip('xyz',center)},**{f'size_{a}':float(v) for a,v in zip('xyz',size)},'exhaustiveness':32,'num_modes':20,'energy_range':5,'seed':20260906,'cpu':CPU,'out':docked}
config.write_text(''.join(f'{k} = {v}\n' for k,v in params.items()))
command([VINA,'--config',config],RUN/'logs'/'vina.log')
if not docked.is_file(): raise RuntimeError('Vina produced no poses.')
print(docked)

## 9. Evaluate all poses
The general-purpose screen reports Vina scores, heavy-atom proximity, clashes below 2 Å, hydrogen bonds using explicit donor hydrogens (D–A ≤3.5 Å, H–A ≤2.6 Å, angle ≥120°), and candidate ionic contacts within 4 Å using user-confirmed ligand charged-group atom serials. These are geometric candidates, not experimental evidence. The generic screen uses AutoDock atom types and standard protein side-chain charge templates; it does not reproduce all of the original taurine-specific rules. Confirm ligand charge groups, protein protonation and atom typing. Absence of explicit H prevents reliable H-bond enumeration. The optional next cell uses the original target-specific taurine evaluation.

In [ ]:
#@title Score and residue-contact analysis
CONTACT_CUTOFF = 4.0
if not math.isfinite(CONTACT_CUTOFF) or CONTACT_CUTOFF<=0: raise ValueError('Invalid distance cutoff.')
def split_poses(text):
    blocks=re.findall(r'^MODEL\s+(\d+)\s*\n(.*?)^ENDMDL[^\n]*',text,re.M|re.S)
    if not blocks: raise ValueError('No Vina MODEL blocks found.')
    result=[]
    for number,body in blocks:
        m=re.search(r'REMARK VINA RESULT:\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s+([-+\d.eE]+)',body)
        if not m: raise ValueError(f'No score for pose {number}')
        result.append((int(number),body,*map(float,m.groups())))
    return result

def heavy_atoms(path):
    # AutoDock atom types H and HD/HS denote hydrogens, not heavy atoms.
    return [(line,xyz) for line,xyz in pdb_atoms(path) if line.split()[-1].upper() not in {'H','HD','HS'}]
ra=heavy_atoms(rec); rpos=np.array([xyz for _,xyz in ra]); residue_rows=[]; score_rows=[]
pose_paths={}
for number,body,score,rmsd_lb,rmsd_ub in split_poses(docked.read_text()):
    p=RUN/'results'/f'pose_{number:03d}.pdbqt'; p.write_text(body); pose_paths[number]=p
    la=heavy_atoms(p); lpos=np.array([xyz for _,xyz in la]); minimum={}
    if not len(ra) or not len(la): raise ValueError('No heavy atoms.')
    for i in range(0,len(ra),512):
        distances=np.linalg.norm(rpos[i:i+512,None,:]-lpos[None,:,:],axis=2).min(axis=1)
        for (line,_),d in zip(ra[i:i+512],distances):
            if d<=CONTACT_CUTOFF:
                key=(line[21:22].strip(),line[22:26].strip(),line[26:27].strip(),line[17:20].strip())
                minimum[key]=min(float(d),minimum.get(key,float('inf')))
    score_rows.append({'pose':number,'vina_score_kcal_mol':score,'vina_rmsd_lb_A':rmsd_lb,'vina_rmsd_ub_A':rmsd_ub,'nearby_residue_count':len(minimum)})
    for (chain,resid,icode,resname),distance in sorted(minimum.items(),key=lambda item:item[1]):
        residue_rows.append({'pose':number,'chain':chain,'residue_number':resid,'insertion_code':icode,'residue_name':resname,'minimum_heavy_atom_distance_A':distance})
scores=pd.DataFrame(score_rows).sort_values(['vina_score_kcal_mol','pose']).reset_index(drop=True)
contacts=pd.DataFrame(residue_rows,columns=['pose','chain','residue_number','insertion_code','residue_name','minimum_heavy_atom_distance_A'])
scores.to_csv(RUN/'results'/'docking_scores.csv',index=False); contacts.to_csv(RUN/'results'/'residue_proximity.csv',index=False)
(RUN/'results'/'contact_settings.json').write_text(json.dumps({'cutoff_A':CONTACT_CUTOFF,'definition':'minimum receptor–ligand heavy-atom distance'},indent=2))
display(scores); display(contacts[contacts.pose==int(scores.iloc[0].pose)])
# Generic interaction screen: explicit hydrogen geometry and charged groups.
# Specify charged ligand atoms by PDBQT serial, including delocalized group atoms.
POSITIVE_LIGAND_SERIALS = "" #@param {type:"string"}
NEGATIVE_LIGAND_SERIALS = "" #@param {type:"string"}
AUTO_TAURINE_CHARGES = False #@param {type:"boolean"}
def parse_serials(text): return {int(s.strip()) for s in text.split(',') if s.strip()}
def atoms(path):
    return [{'serial':int(l[6:11]),'name':l[12:16].strip(),'res':l[17:20].strip(),'label':f'{l[21:22]}:{l[17:20].strip()}{l[22:26].strip()}{l[26:27].strip()}:{l[12:16].strip()}','type':l.split()[-1].upper(),'xyz':np.array(x)} for l,x in pdb_atoms(path)]
def dist(a,b): return float(np.linalg.norm(a['xyz']-b['xyz']))
def dha(d,h,a):
    u=d['xyz']-h['xyz']; w=a['xyz']-h['xyz']; den=np.linalg.norm(u)*np.linalg.norm(w)
    return float(np.degrees(np.arccos(np.clip(np.dot(u,w)/den,-1,1)))) if den else 0.
H_TYPES={'H','HD','HS'}; ACCEPTORS={'NA','OA','SA','NS','OS'}
def donor_hydrogens(items):
    heavy=[a for a in items if a['type'] not in H_TYPES]
    out=[]
    for h in (a for a in items if a['type'] in {'HD','HS'}):
        if not heavy: continue
        d=min(heavy,key=lambda a:dist(a,h))
        if d['type'][:1] in {'N','O','S'} and dist(d,h)<=1.35: out.append((d,h))
    return out
def hbond_pairs(donors,acceptors):
    for d,h in donors:
        for a in acceptors:
            da=dist(d,a)
            if da<=3.5 and dist(h,a)<=2.6 and dha(d,h,a)>=120: yield d,h,a,da
P=atoms(rec); PH=[a for a in P if a['type'] not in H_TYPES]
ppos=[a for a in PH if (a['res'],a['name']) in {('LYS','NZ'),('ARG','NE'),('ARG','NH1'),('ARG','NH2'),('HIP','ND1'),('HIP','NE2')}]
pneg=[a for a in PH if (a['res'],a['name']) in {('ASP','OD1'),('ASP','OD2'),('GLU','OE1'),('GLU','OE2')}]
pdon=donor_hydrogens(P); pacc=[a for a in PH if a['type'] in ACCEPTORS]
interaction_rows=[]; summaries=[]
for number,p in pose_paths.items():
    L=atoms(p); LH=[a for a in L if a['type'] not in H_TYPES]
    lpos=parse_serials(POSITIVE_LIGAND_SERIALS); lneg=parse_serials(NEGATIVE_LIGAND_SERIALS)
    if AUTO_TAURINE_CHARGES:
        if not VERIFY_TAURINE or len(LH)!=7 or sum(a['type'].startswith('N') for a in LH)!=1 or sum(a['type'].startswith('O') for a in LH)!=3 or sum(a['type'].startswith('S') for a in LH)!=1: raise ValueError('Taurine charge assignment requires verified zwitterionic taurine.')
        lpos={a['serial'] for a in LH if a['type'].startswith('N')}; lneg={a['serial'] for a in LH if a['type'].startswith('O')}
    if (lpos|lneg)-{a['serial'] for a in LH} or lpos&lneg: raise ValueError('Invalid ligand charge serials.')
    ldon=donor_hydrogens(L); lacc=[a for a in LH if a['type'] in ACCEPTORS]
    local=[]
    for direction,ds,acs in [('protein_to_ligand',pdon,lacc),('ligand_to_protein',ldon,pacc)]:
        for d,h,a,da in hbond_pairs(ds,acs): local.append({'pose':number,'type':'H-bond candidate','direction':direction,'atom_1':d['label'],'atom_2':a['label'],'distance_A':da,'angle_deg':dha(d,h,a)})
    for grp,ids in [(ppos,lneg),(pneg,lpos)]:
        for a in grp:
            for b in LH:
                if b['serial'] in ids and dist(a,b)<=4.0: local.append({'pose':number,'type':'ionic candidate','direction':'protein_ligand','atom_1':a['label'],'atom_2':b['label'],'distance_A':dist(a,b),'angle_deg':None})
    for a in PH:
        for b in LH:
            d=dist(a,b)
            if d<2.0: local.append({'pose':number,'type':'clash','direction':'protein_ligand','atom_1':a['label'],'atom_2':b['label'],'distance_A':d,'angle_deg':None})
    interaction_rows.extend(local)
    summaries.append({'pose':number,'clashes':sum(x['type']=='clash' for x in local),'hbond_candidates':sum(x['type']=='H-bond candidate' for x in local),'ionic_candidates':sum(x['type']=='ionic candidate' for x in local) if lpos or lneg else None})
interactions=pd.DataFrame(interaction_rows,columns=['pose','type','direction','atom_1','atom_2','distance_A','angle_deg'])
review=scores.merge(pd.DataFrame(summaries),on='pose')
review.to_csv(RUN/'results'/'pose_evaluation.csv',index=False)
interactions.to_csv(RUN/'results'/'interaction_candidates.csv',index=False)
(RUN/'results'/'interaction_settings.json').write_text(json.dumps({'clash_A_strictly_below':2,'hbond_DA_A':3.5,'hbond_HA_A':2.6,'hbond_angle_deg':120,'ionic_A':4,'positive_ligand_serials':POSITIVE_LIGAND_SERIALS,'negative_ligand_serials':NEGATIVE_LIGAND_SERIALS,'auto_taurine':AUTO_TAURINE_CHARGES},indent=2))
display(review)
if not POSITIVE_LIGAND_SERIALS and not NEGATIVE_LIGAND_SERIALS and not AUTO_TAURINE_CHARGES: print('Ionic contacts not evaluated; enter confirmed charged-group atom serials.')


### Optional: original taurine-specific pose evaluation
For reproduction only, run the next cell with the correct original analysis receptor PDB and target. The original scripts require 20 poses; if Vina returns fewer, do not fabricate missing poses. The general-purpose screen above is not the same as the original target-specific analysis. The original functions are included below for transparency.

In [ ]:
# Shared settings and original interaction-analysis implementations.
from pathlib import Path
import re, math, json, shutil, subprocess, sys, datetime, hashlib
import numpy as np
import pandas as pd

TARGETS = {
    "MAPK3": {"center": (28.616, 7.003, 18.087), "size": (20.,20.,22.), "cpu": 0, "published_pose": 2, "published_score": -3.506, "ligand_tool": "Meeko"},
    "EGFR": {"center": (18.866,35.252,37.650), "size": (22.,22.,23.2), "cpu": 0, "published_pose": 3, "published_score": -4.227, "ligand_tool": "Meeko"},
    "STAT5A": {"center": (22.435,59.146,20.009), "size": (20.332,26.288,25.497), "cpu": 8, "published_pose": 2, "published_score": -3.913, "ligand_tool": "Meeko"},
    "STAT5B": {"center": (29.244,8.171,91.108), "size": (22.,22.,22.), "cpu": 0, "published_pose": 4, "published_score": -3.554, "ligand_tool": "Open Babel"},
}
EXHAUSTIVENESS, NUM_MODES, ENERGY_RANGE, SEED = 32, 20, 5, 20260906
CONTACT_MAX, CLASH_MAX, HBOND_DA_MAX, HBOND_HA_MAX, HBOND_ANGLE_MIN, IONIC_MAX = 4.0, 2.0, 3.5, 2.6, 120.0, 4.0

def analyze_mapk3(RECEPTOR_PDB, DOCKING_PDBQT, OUTPUT_DIR):
    """Original MAPK3 pose analysis, with file paths supplied by the caller."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CSV_OUTPUT = OUTPUT_DIR / "MAPK3_all_pose_interaction_ranking.csv"
    TEXT_OUTPUT = OUTPUT_DIR / "MAPK3_all_pose_interaction_report.txt"
    def distance(a, b):
        return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))


    def angle_degrees(a, b, c):
        ba = np.asarray(a) - np.asarray(b)
        bc = np.asarray(c) - np.asarray(b)

        denominator = np.linalg.norm(ba) * np.linalg.norm(bc)

        if denominator == 0:
            return float("nan")

        cosine = np.dot(ba, bc) / denominator
        cosine = np.clip(cosine, -1.0, 1.0)

        return float(np.degrees(np.arccos(cosine)))


    def pdb_element(line):
        element = line[76:78].strip().upper()

        if element:
            return element

        name = line[12:16].strip()
        letters = "".join(character for character in name if character.isalpha())

        return letters[0].upper() if letters else ""


    def pdbqt_element(atom_type):
        atom_type = atom_type.upper()

        mapping = {
            "C": "C", "A": "C",
            "N": "N", "NA": "N", "NS": "N",
            "O": "O", "OA": "O", "OS": "O",
            "S": "S", "SA": "S",
            "H": "H", "HD": "H", "HS": "H"
        }

        return mapping.get(atom_type, atom_type[0])


    def protein_label(atom):
        return (
            f'{atom["chain"]}:{atom["resname"]}'
            f'{atom["resid"]}:{atom["name"]}'
        )


    def ligand_label(atom):
        return f'{atom["name"]}{atom["serial"]}'


    # ================================================================
    # VALIDATE PATHS
    # ================================================================

    if not RECEPTOR_PDB.exists():
        raise FileNotFoundError(
            f"MAPK3 receptor not found:\n{RECEPTOR_PDB}"
        )

    if not DOCKING_PDBQT.exists():
        raise FileNotFoundError(
            f"MAPK3 docking output not found:\n{DOCKING_PDBQT}"
        )

    print("✓ Receptor:", RECEPTOR_PDB)
    print("✓ Docking output:", DOCKING_PDBQT)


    # ================================================================
    # READ MAPK3 RECEPTOR
    # ================================================================

    protein_atoms = []

    with open(RECEPTOR_PDB, "r") as handle:
        for line in handle:

            if not line.startswith(("ATOM  ", "HETATM")):
                continue

            resname = line[17:20].strip().upper()

            if resname in {"HOH", "WAT", "SOL"}:
                continue

            try:
                protein_atoms.append({
                    "name": line[12:16].strip(),
                    "resname": resname,
                    "chain": line[21].strip() or "_",
                    "resid": int(line[22:26]),
                    "element": pdb_element(line),
                    "coord": np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                })
            except ValueError:
                continue

    protein_heavy = [
        atom for atom in protein_atoms
        if atom["element"] not in {"H", "D"}
    ]

    print("Protein heavy atoms:", len(protein_heavy))


    # ================================================================
    # READ ALL DOCKED POSES
    # ================================================================

    poses = []
    inside_model = False
    current_atoms = []
    current_score = None

    with open(DOCKING_PDBQT, "r") as handle:
        for line in handle:

            if line.startswith("MODEL"):
                inside_model = True
                current_atoms = []
                current_score = None

            elif inside_model and "REMARK VINA RESULT:" in line:
                match = re.search(
                    r"REMARK VINA RESULT:\s+(-?\d+(?:\.\d+)?)",
                    line
                )

                if match:
                    current_score = float(match.group(1))

            elif inside_model and line.startswith(("ATOM  ", "HETATM")):
                try:
                    atom_type = line.split()[-1]

                    current_atoms.append({
                        "serial": int(line[6:11]),
                        "name": line[12:16].strip(),
                        "element": pdbqt_element(atom_type),
                        "atom_type": atom_type,
                        "coord": np.array([
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54])
                        ])
                    })
                except (ValueError, IndexError):
                    continue

            elif inside_model and line.startswith("ENDMDL"):
                poses.append({
                    "score": current_score,
                    "atoms": current_atoms
                })

                inside_model = False

    if len(poses) != 20:
        raise RuntimeError(
            f"Expected 20 poses but found {len(poses)}."
        )

    print("Docking poses:", len(poses))


    # ================================================================
    # CHEMICAL GROUP DEFINITIONS
    # ================================================================

    acceptors = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"},
        "ASN": {"OD1"},
        "GLN": {"OE1"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1", "NE2"},
        "HIE": {"ND1", "NE2"},
        "MET": {"SD"},
        "CYS": {"SG"}
    }

    donors = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "ASN": {"ND2"},
        "GLN": {"NE2"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "TRP": {"NE1"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1"},
        "HIE": {"NE2"},
        "HIP": {"ND1", "NE2"},
        "CYS": {"SG"}
    }

    positive_groups = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "HIP": {"ND1", "NE2"}
    }

    negative_groups = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"}
    }


    # ================================================================
    # ANALYZE EACH POSE
    # ================================================================

    results = []
    report_sections = []

    for pose_number, pose in enumerate(poses, start=1):

        atoms = pose["atoms"]

        heavy = [atom for atom in atoms if atom["element"] != "H"]
        hydrogens = [atom for atom in atoms if atom["element"] == "H"]
        nitrogens = [atom for atom in atoms if atom["element"] == "N"]
        sulfurs = [atom for atom in atoms if atom["element"] == "S"]
        oxygens = [atom for atom in atoms if atom["element"] == "O"]

        if len(heavy) != 7 or len(nitrogens) != 1 or len(sulfurs) != 1:
            raise RuntimeError(
                f"Pose {pose_number} has an invalid taurine atom set."
            )

        taurine_n = nitrogens[0]
        taurine_s = sulfurs[0]

        n_hydrogens = [
            hydrogen for hydrogen in hydrogens
            if distance(hydrogen["coord"], taurine_n["coord"]) <= 1.30
        ]

        sulfonate_oxygens = [
            oxygen for oxygen in oxygens
            if distance(oxygen["coord"], taurine_s["coord"]) <= 1.90
        ]

        nearby = {}
        pairs = []
        clashes = []

        for protein_atom in protein_heavy:
            for ligand_atom in heavy:

                d = distance(
                    protein_atom["coord"],
                    ligand_atom["coord"]
                )

                pairs.append((d, protein_atom, ligand_atom))

                if d <= CONTACT_MAX:
                    key = (
                        protein_atom["chain"],
                        protein_atom["resname"],
                        protein_atom["resid"]
                    )

                    nearby[key] = min(
                        nearby.get(key, float("inf")),
                        d
                    )

                if d < CLASH_MAX:
                    clashes.append(
                        (
                            protein_label(protein_atom),
                            ligand_label(ligand_atom),
                            d
                        )
                    )

        pairs.sort(key=lambda item: item[0])
        minimum_distance, closest_protein, closest_ligand = pairs[0]

        # Taurine-donor hydrogen bonds
        hbonds = []

        for acceptor in protein_heavy:

            is_acceptor = (
                acceptor["name"] in {"O", "OXT"}
                or acceptor["name"] in acceptors.get(
                    acceptor["resname"], set()
                )
            )

            if not is_acceptor:
                continue

            donor_acceptor = distance(
                taurine_n["coord"],
                acceptor["coord"]
            )

            if donor_acceptor > HBOND_DA_MAX:
                continue

            for hydrogen in n_hydrogens:

                hydrogen_acceptor = distance(
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                angle = angle_degrees(
                    taurine_n["coord"],
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                if (
                    hydrogen_acceptor <= HBOND_HA_MAX
                    and angle >= HBOND_ANGLE_MIN
                ):
                    hbonds.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}–'
                            f'{ligand_label(hydrogen)}···'
                            f'{protein_label(acceptor)}'
                        ),
                        "da": donor_acceptor,
                        "ha": hydrogen_acceptor,
                        "angle": angle
                    })

        # Preliminary protein-donor contacts
        donor_contacts = []

        for donor in protein_heavy:

            if donor["name"] not in donors.get(
                donor["resname"], set()
            ):
                continue

            for oxygen in sulfonate_oxygens:

                d = distance(
                    donor["coord"],
                    oxygen["coord"]
                )

                if d <= HBOND_DA_MAX:
                    donor_contacts.append({
                        "interaction": (
                            f'{protein_label(donor)}···'
                            f'{ligand_label(oxygen)}'
                        ),
                        "distance": d
                    })

        # Ionic contacts
        ionic_contacts = []

        for protein_atom in protein_heavy:

            if protein_atom["name"] in positive_groups.get(
                protein_atom["resname"], set()
            ):
                for oxygen in sulfonate_oxygens:

                    d = distance(
                        protein_atom["coord"],
                        oxygen["coord"]
                    )

                    if d <= IONIC_MAX:
                        ionic_contacts.append({
                            "interaction": (
                                f'{ligand_label(oxygen)}···'
                                f'{protein_label(protein_atom)}'
                            ),
                            "distance": d,
                            "type": "sulfonate(−)···protein(+)"
                        })

            if protein_atom["name"] in negative_groups.get(
                protein_atom["resname"], set()
            ):
                d = distance(
                    taurine_n["coord"],
                    protein_atom["coord"]
                )

                if d <= IONIC_MAX:
                    ionic_contacts.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}···'
                            f'{protein_label(protein_atom)}'
                        ),
                        "distance": d,
                        "type": "ammonium(+)···protein(−)"
                    })

        residue_labels = [
            f"{chain}:{resname}{resid}"
            for chain, resname, resid in sorted(
                nearby,
                key=lambda key: nearby[key]
            )
        ]

        if clashes:
            interaction_class = "REJECT_CLASH"
            class_rank = 4
        elif hbonds or ionic_contacts:
            interaction_class = "SPECIFIC_INTERACTION"
            class_rank = 1
        elif donor_contacts:
            interaction_class = "PRELIMINARY_POLAR"
            class_rank = 2
        else:
            interaction_class = "CONTACT_ONLY"
            class_rank = 3

        hbond_text = "; ".join(
            item["interaction"] for item in hbonds
        ) or "None"

        donor_text = "; ".join(
            item["interaction"] for item in donor_contacts
        ) or "None"

        ionic_text = "; ".join(
            item["interaction"] for item in ionic_contacts
        ) or "None"

        results.append({
            "Pose": pose_number,
            "Vina_score": pose["score"],
            "Minimum_distance_A": minimum_distance,
            "Clashes": len(clashes),
            "Confirmed_H_bonds": len(hbonds),
            "Preliminary_donor_contacts": len(donor_contacts),
            "Ionic_contacts": len(ionic_contacts),
            "Interaction_class": interaction_class,
            "Class_rank": class_rank,
            "Nearby_residues": "; ".join(residue_labels),
            "H_bond_details": hbond_text,
            "Preliminary_contact_details": donor_text,
            "Ionic_details": ionic_text
        })

        section = [
            "=" * 105,
            (
                f"POSE {pose_number:2d} | "
                f"Vina {pose['score']:.3f} kcal/mol | "
                f"minimum {minimum_distance:.2f} Å | "
                f"clashes {len(clashes)}"
            ),
            "-" * 105,
            "Residues: " + ("; ".join(residue_labels) or "None")
        ]

        if hbonds:
            section.append("Confirmed taurine-donor H-bonds:")

            for item in hbonds:
                section.append(
                    f'  {item["interaction"]} | '
                    f'D–A {item["da"]:.2f} Å | '
                    f'H–A {item["ha"]:.2f} Å | '
                    f'angle {item["angle"]:.1f}°'
                )
        else:
            section.append("Confirmed taurine-donor H-bonds: None")

        if donor_contacts:
            section.append("Preliminary protein-donor contacts:")

            for item in donor_contacts:
                section.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å'
                )
        else:
            section.append("Preliminary protein-donor contacts: None")

        if ionic_contacts:
            section.append("Ionic contacts:")

            for item in ionic_contacts:
                section.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å | '
                    f'{item["type"]}'
                )
        else:
            section.append("Ionic contacts: None")

        report_sections.append("\n".join(section))


    # ================================================================
    # RANK AND SAVE
    # ================================================================

    table = pd.DataFrame(results)

    ranked = table.sort_values(
        by=["Clashes", "Class_rank", "Vina_score"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    ranked.insert(
        0,
        "Interaction_rank",
        range(1, len(ranked) + 1)
    )

    ranked.to_csv(CSV_OUTPUT, index=False)

    with open(TEXT_OUTPUT, "w") as handle:
        handle.write(
            "MAPK3–TAURINE ALL-POSE INTERACTION ANALYSIS\n"
        )
        handle.write("=" * 105 + "\n\n")
        handle.write("\n\n".join(report_sections))
        handle.write("\n\n" + ranked.to_string(index=False))
        handle.write("\n")


    # ================================================================
    # PRINT RESULTS
    # ================================================================

    summary_columns = [
        "Interaction_rank",
        "Pose",
        "Vina_score",
        "Minimum_distance_A",
        "Clashes",
        "Confirmed_H_bonds",
        "Preliminary_donor_contacts",
        "Ionic_contacts",
        "Interaction_class"
    ]

    print("\n" + "=" * 110)
    print("MAPK3–TAURINE INTERACTION-BASED POSE RANKING")
    print("=" * 110)

    print(
        ranked[summary_columns].to_string(index=False)
    )

    best = ranked.iloc[0]

    print("\n" + "=" * 110)
    print("TOP INTERACTION-RANKED POSE")
    print("=" * 110)
    print("Pose:", int(best["Pose"]))
    print("Vina score:", best["Vina_score"], "kcal/mol")
    print("Minimum distance:", f'{best["Minimum_distance_A"]:.2f} Å')
    print("Confirmed H-bonds:", int(best["Confirmed_H_bonds"]))
    print(
        "Preliminary donor contacts:",
        int(best["Preliminary_donor_contacts"])
    )
    print("Ionic contacts:", int(best["Ionic_contacts"]))
    print("Clashes:", int(best["Clashes"]))
    print("Class:", best["Interaction_class"])
    print("Residues:", best["Nearby_residues"])
    print("H-bonds:", best["H_bond_details"])
    print("Ionic contacts:", best["Ionic_details"])

    print("\nSaved:")
    print("✓", CSV_OUTPUT)
    print("✓", TEXT_OUTPUT)
    return ranked

def analyze_egfr(RECEPTOR_PDB, DOCKING_PDBQT, OUTPUT_DIR):
    """Original EGFR pose analysis, with file paths supplied by the caller."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CSV_OUTPUT = OUTPUT_DIR / "EGFR_all_pose_interaction_ranking.csv"
    TEXT_OUTPUT = OUTPUT_DIR / "EGFR_all_pose_interaction_report.txt"
    def distance(a, b):
        return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))


    def angle_degrees(a, b, c):
        ba = np.asarray(a) - np.asarray(b)
        bc = np.asarray(c) - np.asarray(b)

        denominator = np.linalg.norm(ba) * np.linalg.norm(bc)

        if denominator == 0:
            return float("nan")

        cosine = np.dot(ba, bc) / denominator
        cosine = np.clip(cosine, -1.0, 1.0)

        return float(np.degrees(np.arccos(cosine)))


    def pdb_element(line):
        element = line[76:78].strip().upper()

        if element:
            return element

        name = line[12:16].strip()
        letters = "".join(c for c in name if c.isalpha())

        return letters[0].upper() if letters else ""


    def pdbqt_element(atom_type):
        atom_type = atom_type.upper()

        mapping = {
            "C": "C", "A": "C",
            "N": "N", "NA": "N", "NS": "N",
            "O": "O", "OA": "O", "OS": "O",
            "S": "S", "SA": "S",
            "H": "H", "HD": "H", "HS": "H"
        }

        return mapping.get(atom_type, atom_type[0])


    def protein_label(atom):
        return (
            f'{atom["chain"]}:{atom["resname"]}'
            f'{atom["resid"]}:{atom["name"]}'
        )


    def ligand_label(atom):
        return f'{atom["name"]}{atom["serial"]}'


    # ================================================================
    # READ RECEPTOR
    # ================================================================

    for file in [RECEPTOR_PDB, DOCKING_PDBQT]:
        if not file.exists():
            raise FileNotFoundError(file)

    protein = []

    with open(RECEPTOR_PDB) as handle:
        for line in handle:

            if not line.startswith(("ATOM  ", "HETATM")):
                continue

            resname = line[17:20].strip().upper()

            if resname in {"HOH", "WAT", "SOL"}:
                continue

            try:
                protein.append({
                    "name": line[12:16].strip(),
                    "resname": resname,
                    "chain": line[21].strip() or "_",
                    "resid": int(line[22:26]),
                    "element": pdb_element(line),
                    "coord": np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                })
            except ValueError:
                continue

    protein = [
        atom for atom in protein
        if atom["element"] not in {"H", "D"}
    ]

    print("✓ Receptor:", RECEPTOR_PDB)
    print("Protein heavy atoms:", len(protein))


    # ================================================================
    # READ ALL PDBQT POSES
    # ================================================================

    poses = []
    inside = False
    atoms = []
    score = None

    with open(DOCKING_PDBQT) as handle:
        for line in handle:

            if line.startswith("MODEL"):
                inside = True
                atoms = []
                score = None

            elif inside and "REMARK VINA RESULT:" in line:
                match = re.search(
                    r"REMARK VINA RESULT:\s+(-?\d+(?:\.\d+)?)",
                    line
                )

                if match:
                    score = float(match.group(1))

            elif inside and line.startswith(("ATOM  ", "HETATM")):
                try:
                    atom_type = line.split()[-1]

                    atoms.append({
                        "serial": int(line[6:11]),
                        "name": line[12:16].strip(),
                        "element": pdbqt_element(atom_type),
                        "coord": np.array([
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54])
                        ])
                    })
                except (ValueError, IndexError):
                    continue

            elif inside and line.startswith("ENDMDL"):
                poses.append({"score": score, "atoms": atoms})
                inside = False

    if len(poses) != 20:
        raise RuntimeError(f"Expected 20 poses, found {len(poses)}.")

    print("Docking poses:", len(poses))


    # ================================================================
    # INTERACTION ATOM DEFINITIONS
    # ================================================================

    acceptors = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"},
        "ASN": {"OD1"},
        "GLN": {"OE1"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1", "NE2"},
        "HIE": {"ND1", "NE2"},
        "CYS": {"SG"},
        "MET": {"SD"}
    }

    donors = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "ASN": {"ND2"},
        "GLN": {"NE2"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "TRP": {"NE1"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1"},
        "HIE": {"NE2"},
        "HIP": {"ND1", "NE2"},
        "CYS": {"SG"}
    }

    positive = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "HIP": {"ND1", "NE2"}
    }

    negative = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"}
    }


    # ================================================================
    # ANALYZE POSES
    # ================================================================

    rows = []
    sections = []

    for pose_number, pose in enumerate(poses, start=1):

        ligand = pose["atoms"]

        heavy = [a for a in ligand if a["element"] != "H"]
        hydrogens = [a for a in ligand if a["element"] == "H"]
        nitrogen = next(a for a in ligand if a["element"] == "N")
        sulfur = next(a for a in ligand if a["element"] == "S")

        sulfonate_oxygens = [
            a for a in ligand
            if a["element"] == "O"
            and distance(a["coord"], sulfur["coord"]) <= 1.90
        ]

        n_hydrogens = [
            a for a in hydrogens
            if distance(a["coord"], nitrogen["coord"]) <= 1.30
        ]

        if (
            len(heavy) != 7
            or len(sulfonate_oxygens) != 3
            or len(n_hydrogens) != 3
        ):
            raise RuntimeError(
                f"Pose {pose_number}: incomplete taurine atom set."
            )

        nearby = {}
        pairs = []
        clashes = []

        for pa in protein:
            for la in heavy:

                d = distance(pa["coord"], la["coord"])
                pairs.append((d, pa, la))

                if d <= CONTACT_MAX:
                    key = (pa["chain"], pa["resname"], pa["resid"])
                    nearby[key] = min(
                        nearby.get(key, float("inf")),
                        d
                    )

                if d < CLASH_MAX:
                    clashes.append((protein_label(pa), ligand_label(la), d))

        pairs.sort(key=lambda item: item[0])
        minimum_distance, closest_protein, closest_ligand = pairs[0]

        # Taurine donor hydrogen bonds
        hbonds = []

        for acceptor in protein:

            valid_acceptor = (
                acceptor["name"] in {"O", "OXT"}
                or acceptor["name"] in acceptors.get(
                    acceptor["resname"], set()
                )
            )

            if not valid_acceptor:
                continue

            da = distance(nitrogen["coord"], acceptor["coord"])

            if da > HBOND_DA_MAX:
                continue

            for hydrogen in n_hydrogens:

                ha = distance(hydrogen["coord"], acceptor["coord"])

                angle = angle_degrees(
                    nitrogen["coord"],
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                if ha <= HBOND_HA_MAX and angle >= HBOND_ANGLE_MIN:
                    hbonds.append({
                        "interaction": (
                            f'{ligand_label(nitrogen)}–'
                            f'{ligand_label(hydrogen)}···'
                            f'{protein_label(acceptor)}'
                        ),
                        "da": da,
                        "ha": ha,
                        "angle": angle
                    })

        # Protein donor to sulfonate contacts
        donor_contacts = []

        for donor in protein:

            if donor["name"] not in donors.get(
                donor["resname"], set()
            ):
                continue

            for oxygen in sulfonate_oxygens:

                d = distance(donor["coord"], oxygen["coord"])

                if d <= HBOND_DA_MAX:
                    donor_contacts.append({
                        "interaction": (
                            f'{protein_label(donor)}···'
                            f'{ligand_label(oxygen)}'
                        ),
                        "distance": d
                    })

        # Ionic contacts
        ionic = []

        for pa in protein:

            if pa["name"] in positive.get(pa["resname"], set()):

                for oxygen in sulfonate_oxygens:

                    d = distance(pa["coord"], oxygen["coord"])

                    if d <= IONIC_MAX:
                        ionic.append({
                            "interaction": (
                                f'{ligand_label(oxygen)}···'
                                f'{protein_label(pa)}'
                            ),
                            "distance": d,
                            "type": "sulfonate(−)···protein(+)"
                        })

            if pa["name"] in negative.get(pa["resname"], set()):

                d = distance(nitrogen["coord"], pa["coord"])

                if d <= IONIC_MAX:
                    ionic.append({
                        "interaction": (
                            f'{ligand_label(nitrogen)}···'
                            f'{protein_label(pa)}'
                        ),
                        "distance": d,
                        "type": "ammonium(+)···protein(−)"
                    })

        residues = [
            f"{chain}:{resname}{resid}"
            for chain, resname, resid in sorted(
                nearby,
                key=lambda key: nearby[key]
            )
        ]

        if clashes:
            interaction_class = "REJECT_CLASH"
            class_rank = 4
        elif hbonds or ionic:
            interaction_class = "SPECIFIC_INTERACTION"
            class_rank = 1
        elif donor_contacts:
            interaction_class = "PRELIMINARY_POLAR"
            class_rank = 2
        else:
            interaction_class = "CONTACT_ONLY"
            class_rank = 3

        hbond_text = "; ".join(
            item["interaction"] for item in hbonds
        ) or "None"

        donor_text = "; ".join(
            item["interaction"] for item in donor_contacts
        ) or "None"

        ionic_text = "; ".join(
            item["interaction"] for item in ionic
        ) or "None"

        rows.append({
            "Pose": pose_number,
            "Vina_score": pose["score"],
            "Minimum_distance_A": minimum_distance,
            "Clashes": len(clashes),
            "Confirmed_H_bonds": len(hbonds),
            "Preliminary_donor_contacts": len(donor_contacts),
            "Ionic_contacts": len(ionic),
            "Interaction_class": interaction_class,
            "Class_rank": class_rank,
            "Nearby_residues": "; ".join(residues),
            "H_bond_details": hbond_text,
            "Preliminary_contact_details": donor_text,
            "Ionic_details": ionic_text
        })

        lines = [
            "=" * 105,
            (
                f"POSE {pose_number:2d} | "
                f"Vina {pose['score']:.3f} kcal/mol | "
                f"minimum {minimum_distance:.2f} Å | "
                f"clashes {len(clashes)}"
            ),
            "-" * 105,
            "Residues: " + ("; ".join(residues) or "None")
        ]

        if hbonds:
            lines.append("Confirmed taurine-donor H-bonds:")

            for item in hbonds:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'D–A {item["da"]:.2f} Å | '
                    f'H–A {item["ha"]:.2f} Å | '
                    f'angle {item["angle"]:.1f}°'
                )
        else:
            lines.append("Confirmed taurine-donor H-bonds: None")

        if donor_contacts:
            lines.append("Preliminary protein-donor contacts:")

            for item in donor_contacts:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å'
                )
        else:
            lines.append("Preliminary protein-donor contacts: None")

        if ionic:
            lines.append("Ionic contacts:")

            for item in ionic:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å | '
                    f'{item["type"]}'
                )
        else:
            lines.append("Ionic contacts: None")

        sections.append("\n".join(lines))


    # ================================================================
    # RANK, SAVE AND DISPLAY
    # ================================================================

    table = pd.DataFrame(rows)

    ranked = table.sort_values(
        by=["Clashes", "Class_rank", "Vina_score"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    ranked.insert(
        0,
        "Interaction_rank",
        range(1, len(ranked) + 1)
    )

    ranked.to_csv(CSV_OUTPUT, index=False)

    with open(TEXT_OUTPUT, "w") as handle:
        handle.write(
            "EGFR–TAURINE ALL-POSE INTERACTION ANALYSIS\n"
        )
        handle.write("=" * 105 + "\n\n")
        handle.write("\n\n".join(sections))
        handle.write("\n\n" + ranked.to_string(index=False))
        handle.write("\n")

    columns = [
        "Interaction_rank",
        "Pose",
        "Vina_score",
        "Minimum_distance_A",
        "Clashes",
        "Confirmed_H_bonds",
        "Preliminary_donor_contacts",
        "Ionic_contacts",
        "Interaction_class"
    ]

    print("\n" + "=" * 110)
    print("EGFR–TAURINE INTERACTION-BASED POSE RANKING")
    print("=" * 110)
    print(ranked[columns].to_string(index=False))

    best = ranked.iloc[0]

    print("\n" + "=" * 110)
    print("TOP INTERACTION-RANKED POSE")
    print("=" * 110)
    print("Pose:", int(best["Pose"]))
    print("Vina score:", best["Vina_score"], "kcal/mol")
    print("Minimum distance:", f'{best["Minimum_distance_A"]:.2f} Å')
    print("Confirmed H-bonds:", int(best["Confirmed_H_bonds"]))
    print(
        "Preliminary donor contacts:",
        int(best["Preliminary_donor_contacts"])
    )
    print("Ionic contacts:", int(best["Ionic_contacts"]))
    print("Clashes:", int(best["Clashes"]))
    print("Class:", best["Interaction_class"])
    print("Residues:", best["Nearby_residues"])
    print("H-bonds:", best["H_bond_details"])
    print("Ionic contacts:", best["Ionic_details"])

    print("\nSaved:")
    print("✓", CSV_OUTPUT)
    print("✓", TEXT_OUTPUT)
    return ranked

def analyze_stat5a(RECEPTOR_PDB, DOCKING_PDBQT, OUTPUT_DIR):
    """Original STAT5A pose analysis, with file paths supplied by the caller."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CSV_OUTPUT = OUTPUT_DIR / "STAT5A_all_pose_interaction_ranking.csv"
    TEXT_OUTPUT = OUTPUT_DIR / "STAT5A_all_pose_interaction_report.txt"
    def distance(coord1, coord2):
        return float(
            np.linalg.norm(np.asarray(coord1) - np.asarray(coord2))
        )


    def angle_degrees(point1, vertex, point3):
        vector1 = np.asarray(point1) - np.asarray(vertex)
        vector2 = np.asarray(point3) - np.asarray(vertex)

        denominator = (
            np.linalg.norm(vector1) * np.linalg.norm(vector2)
        )

        if denominator == 0:
            return float("nan")

        cosine = np.dot(vector1, vector2) / denominator
        cosine = np.clip(cosine, -1.0, 1.0)

        return float(np.degrees(np.arccos(cosine)))


    def pdb_element(line):
        element = line[76:78].strip().upper()

        if element:
            return element

        atom_name = line[12:16].strip()
        letters = "".join(
            character for character in atom_name
            if character.isalpha()
        )

        return letters[0].upper() if letters else ""


    def pdbqt_element(atom_type):
        atom_type = atom_type.upper()

        mapping = {
            "C": "C",
            "A": "C",
            "N": "N",
            "NA": "N",
            "NS": "N",
            "O": "O",
            "OA": "O",
            "OS": "O",
            "S": "S",
            "SA": "S",
            "H": "H",
            "HD": "H",
            "HS": "H"
        }

        return mapping.get(atom_type, atom_type[0])


    def protein_label(atom):
        return (
            f'{atom["chain"]}:{atom["resname"]}'
            f'{atom["resid"]}:{atom["name"]}'
        )


    def residue_label(atom):
        return (
            f'{atom["chain"]}:{atom["resname"]}'
            f'{atom["resid"]}'
        )


    def ligand_label(atom):
        name = atom["name"] if atom["name"] else atom["element"]
        return f'{name}{atom["serial"]}'


    # ================================================================
    # VALIDATE FILES
    # ================================================================

    for required_file in [RECEPTOR_PDB, DOCKING_PDBQT]:
        if not required_file.exists():
            raise FileNotFoundError(required_file)

    print("✓ Receptor:", RECEPTOR_PDB)
    print("✓ Docking output:", DOCKING_PDBQT)


    # ================================================================
    # READ PROTEIN
    # ================================================================

    protein_atoms = []

    with open(RECEPTOR_PDB, "r") as handle:
        for line in handle:

            if not line.startswith(("ATOM  ", "HETATM")):
                continue

            resname = line[17:20].strip().upper()

            if resname in {"HOH", "WAT", "SOL"}:
                continue

            try:
                protein_atoms.append({
                    "name": line[12:16].strip(),
                    "resname": resname,
                    "chain": line[21].strip() or "_",
                    "resid": int(line[22:26]),
                    "element": pdb_element(line),
                    "coord": np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                })
            except ValueError:
                continue

    protein_heavy = [
        atom for atom in protein_atoms
        if atom["element"] not in {"H", "D"}
    ]

    if not protein_heavy:
        raise RuntimeError("No protein heavy atoms were read.")

    print("Protein heavy atoms:", len(protein_heavy))


    # ================================================================
    # READ ALL 20 PDBQT POSES
    # ================================================================

    poses = []
    current_atoms = []
    current_score = None
    inside_model = False

    with open(DOCKING_PDBQT, "r") as handle:
        for line in handle:

            if line.startswith("MODEL"):
                inside_model = True
                current_atoms = []
                current_score = None

            elif inside_model and "REMARK VINA RESULT:" in line:
                match = re.search(
                    r"REMARK VINA RESULT:\s+(-?\d+(?:\.\d+)?)",
                    line
                )

                if match:
                    current_score = float(match.group(1))

            elif inside_model and line.startswith(("ATOM  ", "HETATM")):
                try:
                    atom_type = line.split()[-1]

                    current_atoms.append({
                        "serial": int(line[6:11]),
                        "name": line[12:16].strip(),
                        "atom_type": atom_type,
                        "element": pdbqt_element(atom_type),
                        "coord": np.array([
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54])
                        ])
                    })
                except (ValueError, IndexError):
                    continue

            elif line.startswith("ENDMDL") and inside_model:
                poses.append({
                    "score": current_score,
                    "atoms": current_atoms
                })

                inside_model = False

    if len(poses) != 20:
        raise RuntimeError(
            f"Expected 20 poses but found {len(poses)}."
        )

    print("Docking poses:", len(poses))


    # ================================================================
    # PROTEIN INTERACTION GROUPS
    # ================================================================

    acceptor_atoms = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"},
        "ASN": {"OD1"},
        "GLN": {"OE1"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1", "NE2"},
        "HIE": {"ND1", "NE2"},
        "CYS": {"SG"},
        "MET": {"SD"}
    }

    donor_atoms = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "ASN": {"ND2"},
        "GLN": {"NE2"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "TRP": {"NE1"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1"},
        "HIE": {"NE2"},
        "HIP": {"ND1", "NE2"},
        "CYS": {"SG"}
    }

    positive_atoms = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "HIP": {"ND1", "NE2"}
    }

    negative_atoms = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"}
    }


    # ================================================================
    # ANALYZE ALL POSES
    # ================================================================

    rows = []
    full_report = []

    for pose_number, pose in enumerate(poses, start=1):

        ligand_atoms = pose["atoms"]

        ligand_heavy = [
            atom for atom in ligand_atoms
            if atom["element"] != "H"
        ]

        nitrogens = [
            atom for atom in ligand_atoms
            if atom["element"] == "N"
        ]

        sulfurs = [
            atom for atom in ligand_atoms
            if atom["element"] == "S"
        ]

        oxygens = [
            atom for atom in ligand_atoms
            if atom["element"] == "O"
        ]

        hydrogens = [
            atom for atom in ligand_atoms
            if atom["element"] == "H"
        ]

        if (
            len(nitrogens) != 1
            or len(sulfurs) != 1
            or len(oxygens) != 3
        ):
            raise RuntimeError(
                f"Pose {pose_number}: invalid taurine atom composition."
            )

        taurine_n = nitrogens[0]
        taurine_s = sulfurs[0]

        n_hydrogens = [
            hydrogen for hydrogen in hydrogens
            if distance(
                hydrogen["coord"],
                taurine_n["coord"]
            ) <= 1.30
        ]

        sulfonate_oxygens = [
            oxygen for oxygen in oxygens
            if distance(
                oxygen["coord"],
                taurine_s["coord"]
            ) <= 1.90
        ]

        # Nearby residues, minimum distance and clashes
        nearby = {}
        distance_pairs = []
        clashes = []

        for protein_atom in protein_heavy:
            for ligand_atom in ligand_heavy:

                d = distance(
                    protein_atom["coord"],
                    ligand_atom["coord"]
                )

                distance_pairs.append(
                    (d, protein_atom, ligand_atom)
                )

                if d <= CONTACT_MAX:
                    key = (
                        protein_atom["chain"],
                        protein_atom["resname"],
                        protein_atom["resid"]
                    )

                    nearby[key] = min(
                        nearby.get(key, float("inf")),
                        d
                    )

                if d < CLASH_MAX:
                    clashes.append(
                        (
                            protein_label(protein_atom),
                            ligand_label(ligand_atom),
                            d
                        )
                    )

        distance_pairs.sort(key=lambda item: item[0])

        minimum_distance, closest_protein, closest_ligand = (
            distance_pairs[0]
        )

        # Taurine-donor hydrogen bonds
        confirmed_hbonds = []

        for acceptor in protein_heavy:

            is_backbone_acceptor = (
                acceptor["name"] in {"O", "OXT"}
            )

            is_sidechain_acceptor = (
                acceptor["name"]
                in acceptor_atoms.get(
                    acceptor["resname"],
                    set()
                )
            )

            if not (
                is_backbone_acceptor
                or is_sidechain_acceptor
            ):
                continue

            donor_acceptor = distance(
                taurine_n["coord"],
                acceptor["coord"]
            )

            if donor_acceptor > HBOND_DA_MAX:
                continue

            for hydrogen in n_hydrogens:

                hydrogen_acceptor = distance(
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                angle = angle_degrees(
                    taurine_n["coord"],
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                if (
                    hydrogen_acceptor <= HBOND_HA_MAX
                    and angle >= HBOND_ANGLE_MIN
                ):
                    confirmed_hbonds.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}–'
                            f'{ligand_label(hydrogen)}···'
                            f'{protein_label(acceptor)}'
                        ),
                        "da": donor_acceptor,
                        "ha": hydrogen_acceptor,
                        "angle": angle
                    })

        # Protein-donor heavy-atom contacts
        preliminary_contacts = []

        for donor in protein_heavy:

            if donor["name"] not in donor_atoms.get(
                donor["resname"],
                set()
            ):
                continue

            for oxygen in sulfonate_oxygens:

                d = distance(
                    donor["coord"],
                    oxygen["coord"]
                )

                if d <= HBOND_DA_MAX:
                    preliminary_contacts.append({
                        "interaction": (
                            f'{protein_label(donor)}···'
                            f'{ligand_label(oxygen)}'
                        ),
                        "distance": d
                    })

        # Ionic contacts
        ionic_contacts = []

        for protein_atom in protein_heavy:

            if protein_atom["name"] in positive_atoms.get(
                protein_atom["resname"],
                set()
            ):
                for oxygen in sulfonate_oxygens:

                    d = distance(
                        protein_atom["coord"],
                        oxygen["coord"]
                    )

                    if d <= IONIC_MAX:
                        ionic_contacts.append({
                            "interaction": (
                                f'{ligand_label(oxygen)}···'
                                f'{protein_label(protein_atom)}'
                            ),
                            "distance": d,
                            "type": "sulfonate(−)···protein(+)"
                        })

            if protein_atom["name"] in negative_atoms.get(
                protein_atom["resname"],
                set()
            ):
                d = distance(
                    taurine_n["coord"],
                    protein_atom["coord"]
                )

                if d <= IONIC_MAX:
                    ionic_contacts.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}···'
                            f'{protein_label(protein_atom)}'
                        ),
                        "distance": d,
                        "type": "ammonium(+)···protein(−)"
                    })

        nearby_labels = [
            f"{chain}:{resname}{resid}"
            for chain, resname, resid in sorted(
                nearby,
                key=lambda key: nearby[key]
            )
        ]

        if clashes:
            interaction_class = "REJECT_CLASH"
            class_rank = 4
        elif confirmed_hbonds or ionic_contacts:
            interaction_class = "SPECIFIC_INTERACTION"
            class_rank = 1
        elif preliminary_contacts:
            interaction_class = "PRELIMINARY_POLAR"
            class_rank = 2
        else:
            interaction_class = "CONTACT_ONLY"
            class_rank = 3

        hbond_details = "; ".join(
            item["interaction"]
            for item in confirmed_hbonds
        ) or "None"

        ionic_details = "; ".join(
            item["interaction"]
            for item in ionic_contacts
        ) or "None"

        preliminary_details = "; ".join(
            item["interaction"]
            for item in preliminary_contacts
        ) or "None"

        rows.append({
            "Pose": pose_number,
            "Vina_score": pose["score"],
            "Minimum_distance_A": minimum_distance,
            "Clashes": len(clashes),
            "Confirmed_H_bonds": len(confirmed_hbonds),
            "Preliminary_donor_contacts": len(preliminary_contacts),
            "Ionic_contacts": len(ionic_contacts),
            "Interaction_class": interaction_class,
            "Class_rank": class_rank,
            "Nearby_residues": "; ".join(nearby_labels),
            "H_bond_details": hbond_details,
            "Preliminary_contact_details": preliminary_details,
            "Ionic_details": ionic_details,
            "Closest_pair": (
                f"{protein_label(closest_protein)}—"
                f"{ligand_label(closest_ligand)}"
            )
        })

        lines = [
            "=" * 105,
            (
                f"POSE {pose_number:2d} | "
                f"Vina {pose['score']:.3f} kcal/mol | "
                f"minimum {minimum_distance:.2f} Å | "
                f"clashes {len(clashes)}"
            ),
            "-" * 105,
            "Residues: " + ("; ".join(nearby_labels) or "None")
        ]

        if confirmed_hbonds:
            lines.append("Confirmed taurine-donor H-bonds:")

            for item in confirmed_hbonds:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'D–A {item["da"]:.2f} Å | '
                    f'H–A {item["ha"]:.2f} Å | '
                    f'angle {item["angle"]:.1f}°'
                )
        else:
            lines.append("Confirmed taurine-donor H-bonds: None")

        if preliminary_contacts:
            lines.append("Preliminary protein-donor contacts:")

            for item in preliminary_contacts:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å'
                )
        else:
            lines.append("Preliminary protein-donor contacts: None")

        if ionic_contacts:
            lines.append("Ionic contacts:")

            for item in ionic_contacts:
                lines.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å | '
                    f'{item["type"]}'
                )
        else:
            lines.append("Ionic contacts: None")

        full_report.append("\n".join(lines))


    # ================================================================
    # CREATE RANKING
    # ================================================================

    table = pd.DataFrame(rows)

    ranked = table.sort_values(
        by=[
            "Clashes",
            "Class_rank",
            "Vina_score"
        ],
        ascending=[
            True,
            True,
            True
        ]
    ).reset_index(drop=True)

    ranked.insert(
        0,
        "Interaction_rank",
        range(1, len(ranked) + 1)
    )

    ranked.to_csv(CSV_OUTPUT, index=False)

    with open(TEXT_OUTPUT, "w") as handle:
        handle.write(
            "STAT5A–TAURINE ALL-POSE INTERACTION ANALYSIS\n"
        )
        handle.write("=" * 105 + "\n\n")
        handle.write("\n\n".join(full_report))
        handle.write("\n\n" + "=" * 105 + "\n")
        handle.write("INTERACTION-BASED RANKING\n")
        handle.write("=" * 105 + "\n")
        handle.write(ranked.to_string(index=False))
        handle.write("\n")


    # ================================================================
    # DISPLAY RESULTS
    # ================================================================

    summary_columns = [
        "Interaction_rank",
        "Pose",
        "Vina_score",
        "Minimum_distance_A",
        "Clashes",
        "Confirmed_H_bonds",
        "Preliminary_donor_contacts",
        "Ionic_contacts",
        "Interaction_class"
    ]

    print("\n" + "=" * 110)
    print("STAT5A–TAURINE INTERACTION-BASED POSE RANKING")
    print("=" * 110)

    print(
        ranked[summary_columns].to_string(index=False)
    )

    best = ranked.iloc[0]

    print("\n" + "=" * 110)
    print("TOP INTERACTION-RANKED POSE")
    print("=" * 110)
    print("Pose:", int(best["Pose"]))
    print("Vina score:", best["Vina_score"], "kcal/mol")
    print("Minimum distance:", f'{best["Minimum_distance_A"]:.2f} Å')
    print("Confirmed H-bonds:", int(best["Confirmed_H_bonds"]))
    print(
        "Preliminary donor contacts:",
        int(best["Preliminary_donor_contacts"])
    )
    print("Ionic contacts:", int(best["Ionic_contacts"]))
    print("Clashes:", int(best["Clashes"]))
    print("Class:", best["Interaction_class"])
    print("Residues:", best["Nearby_residues"])
    print("H-bonds:", best["H_bond_details"])
    print("Ionic contacts:", best["Ionic_details"])

    print("\nSaved:")
    print("✓", CSV_OUTPUT)
    print("✓", TEXT_OUTPUT)
    return ranked

def analyze_stat5b(RECEPTOR_PDB, DOCKING_PDBQT, OUTPUT_DIR):
    """Original STAT5B pose analysis, with file paths supplied by the caller."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CSV_OUTPUT = OUTPUT_DIR / "STAT5B_all_pose_interaction_ranking.csv"
    TEXT_OUTPUT = OUTPUT_DIR / "STAT5B_all_pose_interaction_report.txt"
    def distance(coord1, coord2):
        return float(
            np.linalg.norm(np.asarray(coord1) - np.asarray(coord2))
        )


    def angle_degrees(point1, vertex, point3):
        vector1 = np.asarray(point1) - np.asarray(vertex)
        vector2 = np.asarray(point3) - np.asarray(vertex)

        denominator = np.linalg.norm(vector1) * np.linalg.norm(vector2)

        if denominator == 0:
            return float("nan")

        cosine = np.dot(vector1, vector2) / denominator
        cosine = np.clip(cosine, -1.0, 1.0)

        return float(np.degrees(np.arccos(cosine)))


    def pdb_element(line):
        element = line[76:78].strip().upper()

        if element:
            return element

        atom_name = line[12:16].strip()
        letters = "".join(
            character for character in atom_name
            if character.isalpha()
        )

        return letters[0].upper() if letters else ""


    def pdbqt_element(atom_type):
        atom_type = atom_type.upper()

        mapping = {
            "C": "C",
            "A": "C",
            "N": "N",
            "NA": "N",
            "NS": "N",
            "OA": "O",
            "OS": "O",
            "O": "O",
            "S": "S",
            "SA": "S",
            "HD": "H",
            "HS": "H",
            "H": "H"
        }

        return mapping.get(atom_type, atom_type[0])


    def protein_label(atom):
        return (
            f'{atom["chain"]}:{atom["resname"]}'
            f'{atom["resid"]}:{atom["name"]}'
        )


    def ligand_label(atom):
        return f'{atom["name"] or atom["element"]}{atom["serial"]}'


    # ================================================================
    # READ PROTEIN
    # ================================================================

    if not RECEPTOR_PDB.exists():
        raise FileNotFoundError(RECEPTOR_PDB)

    protein_atoms = []

    with open(RECEPTOR_PDB, "r") as handle:
        for line in handle:

            if not line.startswith(("ATOM  ", "HETATM")):
                continue

            residue_name = line[17:20].strip().upper()

            if residue_name in {"HOH", "WAT", "SOL"}:
                continue

            try:
                atom = {
                    "name": line[12:16].strip(),
                    "resname": residue_name,
                    "chain": line[21].strip() or "_",
                    "resid": int(line[22:26]),
                    "element": pdb_element(line),
                    "coord": np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                }

                protein_atoms.append(atom)

            except ValueError:
                continue

    protein_heavy = [
        atom for atom in protein_atoms
        if atom["element"] not in {"H", "D"}
    ]

    print("Protein heavy atoms:", len(protein_heavy))


    # ================================================================
    # READ ALL PDBQT POSES, INCLUDING DOCKED POLAR HYDROGENS
    # ================================================================

    if not DOCKING_PDBQT.exists():
        raise FileNotFoundError(DOCKING_PDBQT)

    poses = []
    current_atoms = []
    current_score = None
    inside_model = False

    with open(DOCKING_PDBQT, "r") as handle:
        for line in handle:

            if line.startswith("MODEL"):
                inside_model = True
                current_atoms = []
                current_score = None

            elif "REMARK VINA RESULT:" in line:
                match = re.search(
                    r"REMARK VINA RESULT:\s+(-?\d+(?:\.\d+)?)",
                    line
                )

                if match:
                    current_score = float(match.group(1))

            elif inside_model and line.startswith(("ATOM  ", "HETATM")):
                try:
                    tokens = line.split()
                    atom_type = tokens[-1]

                    current_atoms.append({
                        "serial": int(line[6:11]),
                        "name": line[12:16].strip(),
                        "atom_type": atom_type,
                        "element": pdbqt_element(atom_type),
                        "coord": np.array([
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54])
                        ])
                    })

                except (ValueError, IndexError):
                    continue

            elif line.startswith("ENDMDL"):
                poses.append({
                    "score": current_score,
                    "atoms": current_atoms
                })

                inside_model = False

    if len(poses) != 20:
        raise RuntimeError(
            f"Expected 20 poses but found {len(poses)}."
        )

    print("Docking poses:", len(poses))


    # ================================================================
    # PROTEIN CHEMICAL GROUP DEFINITIONS
    # ================================================================

    protein_acceptors = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"},
        "ASN": {"OD1"},
        "GLN": {"OE1"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1", "NE2"},
        "HIE": {"ND1", "NE2"},
        "MET": {"SD"},
        "CYS": {"SG"}
    }

    protein_donors = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "ASN": {"ND2"},
        "GLN": {"NE2"},
        "SER": {"OG"},
        "THR": {"OG1"},
        "TYR": {"OH"},
        "TRP": {"NE1"},
        "HIS": {"ND1", "NE2"},
        "HID": {"ND1"},
        "HIE": {"NE2"},
        "HIP": {"ND1", "NE2"},
        "CYS": {"SG"}
    }

    positive_groups = {
        "LYS": {"NZ"},
        "ARG": {"NE", "NH1", "NH2"},
        "HIP": {"ND1", "NE2"}
    }

    negative_groups = {
        "ASP": {"OD1", "OD2"},
        "GLU": {"OE1", "OE2"}
    }


    # ================================================================
    # ANALYZE EVERY POSE
    # ================================================================

    results = []
    detailed_reports = []

    for pose_number, pose in enumerate(poses, start=1):

        ligand_atoms = pose["atoms"]

        ligand_heavy = [
            atom for atom in ligand_atoms
            if atom["element"] != "H"
        ]

        nitrogens = [
            atom for atom in ligand_atoms
            if atom["element"] == "N"
        ]

        sulfurs = [
            atom for atom in ligand_atoms
            if atom["element"] == "S"
        ]

        oxygens = [
            atom for atom in ligand_atoms
            if atom["element"] == "O"
        ]

        hydrogens = [
            atom for atom in ligand_atoms
            if atom["element"] == "H"
        ]

        if len(nitrogens) != 1 or len(sulfurs) != 1 or len(oxygens) != 3:
            raise RuntimeError(
                f"Pose {pose_number}: taurine functional groups "
                "could not be identified."
            )

        taurine_n = nitrogens[0]
        taurine_s = sulfurs[0]

        # Identify hydrogens covalently attached to taurine nitrogen
        n_hydrogens = [
            hydrogen for hydrogen in hydrogens
            if distance(
                hydrogen["coord"],
                taurine_n["coord"]
            ) <= 1.30
        ]

        # Identify three sulfonate oxygens bonded to sulfur
        sulfonate_oxygens = [
            oxygen for oxygen in oxygens
            if distance(
                oxygen["coord"],
                taurine_s["coord"]
            ) <= 1.90
        ]

        # ------------------------------------------------------------
        # Nearby residues and clashes
        # ------------------------------------------------------------

        nearby_residues = {}
        clashes = []
        all_pairs = []

        for protein_atom in protein_heavy:
            for ligand_atom in ligand_heavy:
                d = distance(
                    protein_atom["coord"],
                    ligand_atom["coord"]
                )

                all_pairs.append((d, protein_atom, ligand_atom))

                if d <= CONTACT_MAX:
                    residue_key = (
                        protein_atom["chain"],
                        protein_atom["resname"],
                        protein_atom["resid"]
                    )

                    previous = nearby_residues.get(
                        residue_key,
                        float("inf")
                    )

                    nearby_residues[residue_key] = min(previous, d)

                if d < CLASH_MAX:
                    clashes.append(
                        (
                            protein_label(protein_atom),
                            ligand_label(ligand_atom),
                            d
                        )
                    )

        all_pairs.sort(key=lambda item: item[0])
        minimum_distance, closest_protein, closest_ligand = all_pairs[0]

        # ------------------------------------------------------------
        # Taurine N–H donor hydrogen bonds
        # ------------------------------------------------------------

        confirmed_hbonds = []

        for acceptor in protein_heavy:

            is_backbone_acceptor = acceptor["name"] in {"O", "OXT"}

            is_sidechain_acceptor = (
                acceptor["name"]
                in protein_acceptors.get(
                    acceptor["resname"],
                    set()
                )
            )

            if not (is_backbone_acceptor or is_sidechain_acceptor):
                continue

            donor_acceptor = distance(
                taurine_n["coord"],
                acceptor["coord"]
            )

            if donor_acceptor > HBOND_DA_MAX:
                continue

            for hydrogen in n_hydrogens:
                hydrogen_acceptor = distance(
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                angle = angle_degrees(
                    taurine_n["coord"],
                    hydrogen["coord"],
                    acceptor["coord"]
                )

                if (
                    hydrogen_acceptor <= HBOND_HA_MAX
                    and angle >= HBOND_ANGLE_MIN
                ):
                    confirmed_hbonds.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}–'
                            f'{ligand_label(hydrogen)}···'
                            f'{protein_label(acceptor)}'
                        ),
                        "da": donor_acceptor,
                        "ha": hydrogen_acceptor,
                        "angle": angle
                    })

        # ------------------------------------------------------------
        # Protein-donor contacts to taurine sulfonate
        # These remain preliminary because receptor H atoms are absent.
        # ------------------------------------------------------------

        protein_donor_contacts = []

        for donor in protein_heavy:

            if donor["name"] not in protein_donors.get(
                donor["resname"],
                set()
            ):
                continue

            for oxygen in sulfonate_oxygens:
                d = distance(
                    donor["coord"],
                    oxygen["coord"]
                )

                if d <= HBOND_DA_MAX:
                    protein_donor_contacts.append({
                        "interaction": (
                            f'{protein_label(donor)}···'
                            f'{ligand_label(oxygen)}'
                        ),
                        "distance": d
                    })

        # ------------------------------------------------------------
        # Ionic contacts
        # ------------------------------------------------------------

        ionic_contacts = []

        # Protein positive group to taurine sulfonate
        for protein_atom in protein_heavy:

            if protein_atom["name"] in positive_groups.get(
                protein_atom["resname"],
                set()
            ):
                for oxygen in sulfonate_oxygens:
                    d = distance(
                        protein_atom["coord"],
                        oxygen["coord"]
                    )

                    if d <= IONIC_MAX:
                        ionic_contacts.append({
                            "interaction": (
                                f'{ligand_label(oxygen)}···'
                                f'{protein_label(protein_atom)}'
                            ),
                            "distance": d,
                            "type": "sulfonate(−)···protein(+)"
                        })

            # Taurine ammonium to protein negative group
            if protein_atom["name"] in negative_groups.get(
                protein_atom["resname"],
                set()
            ):
                d = distance(
                    taurine_n["coord"],
                    protein_atom["coord"]
                )

                if d <= IONIC_MAX:
                    ionic_contacts.append({
                        "interaction": (
                            f'{ligand_label(taurine_n)}···'
                            f'{protein_label(protein_atom)}'
                        ),
                        "distance": d,
                        "type": "ammonium(+)···protein(−)"
                    })

        # ------------------------------------------------------------
        # Residue labels
        # ------------------------------------------------------------

        residue_labels = [
            f"{chain}:{resname}{resid}"
            for chain, resname, resid in sorted(
                nearby_residues,
                key=lambda key: nearby_residues[key]
            )
        ]

        hbond_text = "; ".join(
            item["interaction"]
            for item in confirmed_hbonds
        ) or "None"

        ionic_text = "; ".join(
            item["interaction"]
            for item in ionic_contacts
        ) or "None"

        donor_text = "; ".join(
            item["interaction"]
            for item in protein_donor_contacts
        ) or "None"

        # Ranking categories
        if clashes:
            interaction_class = "REJECT_CLASH"
            ranking_class = 4
        elif confirmed_hbonds or ionic_contacts:
            interaction_class = "SPECIFIC_INTERACTION"
            ranking_class = 1
        elif protein_donor_contacts:
            interaction_class = "PRELIMINARY_POLAR"
            ranking_class = 2
        else:
            interaction_class = "CONTACT_ONLY"
            ranking_class = 3

        results.append({
            "Pose": pose_number,
            "Vina_score": pose["score"],
            "Minimum_distance_A": minimum_distance,
            "Clashes": len(clashes),
            "Confirmed_H_bonds": len(confirmed_hbonds),
            "Preliminary_donor_contacts": len(protein_donor_contacts),
            "Ionic_contacts": len(ionic_contacts),
            "Interaction_class": interaction_class,
            "Ranking_class": ranking_class,
            "Nearby_residues": "; ".join(residue_labels),
            "H_bond_details": hbond_text,
            "Preliminary_contact_details": donor_text,
            "Ionic_details": ionic_text,
            "Closest_pair": (
                f"{protein_label(closest_protein)}—"
                f"{ligand_label(closest_ligand)}"
            )
        })

        detail = []
        detail.append("=" * 105)
        detail.append(
            f"POSE {pose_number:2d} | "
            f"Vina {pose['score']:.3f} kcal/mol | "
            f"minimum {minimum_distance:.2f} Å | "
            f"clashes {len(clashes)}"
        )
        detail.append("-" * 105)
        detail.append(
            "Residues: " + ("; ".join(residue_labels) or "None")
        )

        if confirmed_hbonds:
            detail.append("Confirmed taurine-donor H-bonds:")

            for item in confirmed_hbonds:
                detail.append(
                    f'  {item["interaction"]} | '
                    f'D–A {item["da"]:.2f} Å | '
                    f'H–A {item["ha"]:.2f} Å | '
                    f'angle {item["angle"]:.1f}°'
                )
        else:
            detail.append("Confirmed taurine-donor H-bonds: None")

        if protein_donor_contacts:
            detail.append("Preliminary protein-donor contacts:")

            for item in protein_donor_contacts:
                detail.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å'
                )
        else:
            detail.append("Preliminary protein-donor contacts: None")

        if ionic_contacts:
            detail.append("Ionic contacts:")

            for item in ionic_contacts:
                detail.append(
                    f'  {item["interaction"]} | '
                    f'{item["distance"]:.2f} Å | '
                    f'{item["type"]}'
                )
        else:
            detail.append("Ionic contacts: None")

        detailed_reports.append("\n".join(detail))


    # ================================================================
    # RANK POSES
    # Priority: no clashes, specific interactions, Vina score
    # ================================================================

    table = pd.DataFrame(results)

    ranked = table.sort_values(
        by=[
            "Clashes",
            "Ranking_class",
            "Vina_score"
        ],
        ascending=[
            True,
            True,
            True
        ]
    ).reset_index(drop=True)

    ranked.insert(0, "Interaction_rank", range(1, len(ranked) + 1))

    ranked.to_csv(CSV_OUTPUT, index=False)

    with open(TEXT_OUTPUT, "w") as handle:
        handle.write(
            "STAT5B–TAURINE ALL-POSE INTERACTION ANALYSIS\n"
        )
        handle.write("=" * 105 + "\n\n")
        handle.write("\n\n".join(detailed_reports))
        handle.write("\n\n" + "=" * 105 + "\n")
        handle.write("INTERACTION-BASED RANKING\n")
        handle.write("=" * 105 + "\n")
        handle.write(
            ranked[
                [
                    "Interaction_rank",
                    "Pose",
                    "Vina_score",
                    "Minimum_distance_A",
                    "Clashes",
                    "Confirmed_H_bonds",
                    "Preliminary_donor_contacts",
                    "Ionic_contacts",
                    "Interaction_class"
                ]
            ].to_string(index=False)
        )
        handle.write("\n")

    print("\n" + "=" * 110)
    print("STAT5B–TAURINE INTERACTION-BASED POSE RANKING")
    print("=" * 110)

    print(
        ranked[
            [
                "Interaction_rank",
                "Pose",
                "Vina_score",
                "Minimum_distance_A",
                "Clashes",
                "Confirmed_H_bonds",
                "Preliminary_donor_contacts",
                "Ionic_contacts",
                "Interaction_class"
            ]
        ].to_string(index=False)
    )

    print("\nSaved:")
    print("✓", CSV_OUTPUT)
    print("✓", TEXT_OUTPUT)

    best = ranked.iloc[0]

    print("\n" + "=" * 110)
    print("TOP INTERACTION-RANKED POSE")
    print("=" * 110)
    print("Pose:", int(best["Pose"]))
    print("Vina score:", best["Vina_score"], "kcal/mol")
    print("Confirmed H-bonds:", int(best["Confirmed_H_bonds"]))
    print("Preliminary donor contacts:", int(best["Preliminary_donor_contacts"]))
    print("Ionic contacts:", int(best["Ionic_contacts"]))
    print("Clashes:", int(best["Clashes"]))
    print("Class:", best["Interaction_class"])
    print("Residues:", best["Nearby_residues"])
    print("H-bonds:", best["H_bond_details"])
    print("Ionic contacts:", best["Ionic_details"])
    return ranked

ANALYZERS = {"MAPK3": analyze_mapk3, "EGFR": analyze_egfr, "STAT5A": analyze_stat5a, "STAT5B": analyze_stat5b}


In [ ]:
#@title Run original taurine pose evaluation (optional)
RUN_ORIGINAL_ANALYSIS = False #@param {type:"boolean"}
ORIGINAL_TARGET = "MAPK3" #@param ["MAPK3","EGFR","STAT5A","STAT5B"]
ORIGINAL_ANALYSIS_PDB_PATH = "" #@param {type:"string"}
if RUN_ORIGINAL_ANALYSIS:
    if BOX_MODE!='Paper preset' or PAPER_TARGET!=ORIGINAL_TARGET: raise ValueError('Use matching original paper preset.')
    if len(pose_paths)!=20: raise ValueError('Original analysis expects 20 poses.')
    original_receptor=get_input('original_analysis_receptor',{'.pdb'},ORIGINAL_ANALYSIS_PDB_PATH)
    original_ranked=ANALYZERS[ORIGINAL_TARGET](original_receptor,docked,RUN/'results'/'original_interaction_analysis')
    display(original_ranked)
else: print('Original taurine analysis skipped.')

## 10. Inspect and select a pose
Use the evaluation table and inspect predicted affinity, clashes and contacts. Do not automatically assume the most negative score is the most plausible pose. Record a rationale for the choice. For the original paper compare against the original interaction report and selected pose.

In [ ]:
#@title Select pose and view complex
POSE_NUMBER = 1 #@param {type:"integer"}
SELECTION_RATIONALE = "" #@param {type:"string"}
if POSE_NUMBER not in pose_paths: raise ValueError(f'Choose a pose from {list(pose_paths)}')
if not SELECTION_RATIONALE.strip(): raise ValueError('Document the reason for pose selection using the evaluation table.')
selected=POSE_NUMBER
shutil.copy2(pose_paths[selected],RUN/'results'/'selected_pose.pdbqt')
(RUN/'results'/'selected_pose.json').write_text(json.dumps({'pose':selected,'selection_rationale':SELECTION_RATIONALE},indent=2))
display(review[review.pose==selected]); display(interactions[interactions.pose==selected]); show_structure(rec,pose_paths[selected])

## 11. Export results
Archive includes input files, prepared structures, individual poses, scores, residue proximity, configuration, preparation logs (when applicable) and version information. Review licensing of input structures before public redistribution.

In [ ]:
#@title Export ZIP and optionally save to Google Drive
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/Docking_Results" #@param {type:"string"}
DOWNLOAD_ZIP = True #@param {type:"boolean"}
zip_path=Path(shutil.make_archive(str(RUN),'zip',root_dir=RUN))
if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    destination=Path(DRIVE_OUTPUT_FOLDER)
    if not str(destination.resolve()).startswith('/content/drive/'): raise ValueError('Use a mounted Drive folder.')
    destination.mkdir(parents=True,exist_ok=True); shutil.copy2(zip_path,destination/zip_path.name)
    print('Saved:',destination/zip_path.name)
print('Archive:',zip_path)
if DOWNLOAD_ZIP: files.download(str(zip_path))